# Лабораторная работа: Oxford-IIIT Pet сегментация

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
%pip -q install tensorflow-datasets
import tensorflow_datasets as tfds
ds=tfds.load('oxford_iiit_pet',split=['train','test'])
def prep(x):
 return tf.image.resize(x['image'],(128,128))/255.,tf.cast(tf.image.resize(x['segmentation_mask'],(128,128),method='nearest')[...,0],tf.int32)-1
train=ds[0].map(prep).shuffle(1000).batch(16).prefetch(tf.data.AUTOTUNE)
test=ds[1].map(prep).batch(16).prefetch(tf.data.AUTOTUNE)
inp=keras.Input((128,128,3));x=layers.Conv2D(32,3,padding='same',activation='relu')(inp)
x=layers.MaxPooling2D()(x);x=layers.Conv2D(64,3,padding='same',activation='relu')(x)
x=layers.MaxPooling2D()(x);x=layers.Conv2D(128,3,padding='same',activation='relu')(x)
x=layers.UpSampling2D()(x);x=layers.Conv2D(64,3,padding='same',activation='relu')(x)
x=layers.UpSampling2D()(x);out=layers.Conv2D(3,1,activation='softmax')(x)
model=keras.Model(inp,out);model.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy'])
history=model.fit(train,validation_data=test,epochs=12)
pd.DataFrame(history.history).plot(grid=True);plt.show()